# Delhi Daily Climate: Data Audit and Temperature Forecast

Run Jupyter from the project root. The source file is `data/daily_climate.csv`; this notebook audits it and builds a cleaned in-memory copy without modifying the source CSV. Duplicate dates, missing calendar dates, missing measurements, and statistical outliers are reported rather than silently dropped or filled.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.evaluation.evaluate import evaluate_arima
from src.models.arima import forecast_temperatures

DATA_PATH = Path("data/daily_climate.csv")
raw_df = pd.read_csv(DATA_PATH)
print(f"Rows and columns: {raw_df.shape}")
display(raw_df.head())
raw_df.info()
display(raw_df.isna().sum().rename("missing values"))
display(raw_df.describe())

## Parse types and check the daily date index

The date column is parsed strictly (invalid dates are reported and stop the analysis), and measurement columns are converted to numeric values. Duplicate dates are not arbitrarily deduplicated. Missing calendar dates are reported; this notebook does not invent measurements for them.

In [ ]:
df = raw_df.copy()
df["date"] = pd.to_datetime(df["date"], errors="coerce")
invalid_dates = raw_df.loc[df["date"].isna()]
print(f"Invalid or missing dates: {len(invalid_dates)}")
if not invalid_dates.empty:
    display(invalid_dates)
    raise ValueError("Fix invalid or missing dates before continuing.")

numeric_columns = [column for column in df.columns if column != "date"]
for column in numeric_columns:
    converted = pd.to_numeric(df[column], errors="coerce")
    invalid_values = df[column].notna() & converted.isna()
    if invalid_values.any():
        raise ValueError(f"Non-numeric values found in {column}: {df.loc[invalid_values, column].tolist()}")
    df[column] = converted

duplicate_dates = df.loc[df["date"].duplicated(keep=False)].sort_values("date")
print(f"Rows with duplicate dates: {len(duplicate_dates)}")
if not duplicate_dates.empty:
    display(duplicate_dates)
    raise ValueError("Resolve duplicate dates before treating rows as daily observations.")

df = df.sort_values("date").set_index("date")
expected_dates = pd.date_range(df.index.min(), df.index.max(), freq="D", name="date")
missing_dates = expected_dates.difference(df.index)
print(f"Missing calendar dates: {len(missing_dates)}")
if len(missing_dates):
    display(missing_dates.to_series(index=range(len(missing_dates)), name="missing date"))

display(df.dtypes.rename("cleaned dtype"))
display(df.isna().sum().rename("missing measurements after type conversion"))

## Statistical outliers and physically implausible values

The 1.5×IQR rule is used to flag unusual values, not to automatically remove them: unusual weather can be genuine. For pressure, values outside 800–1100 hPa are treated as implausible for this dataset and investigated separately. The Jan 1 value of 59.0 is far from all surrounding pressure measurements (about 1,018 hPa); without a source record to establish the intended value, it is marked missing instead of guessed or interpolated.

In [ ]:
numeric_columns = df.select_dtypes(include="number").columns.tolist()
outlier_rows = []
for column in numeric_columns:
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    flagged = df[column].lt(lower) | df[column].gt(upper)
    print(f"{column}: IQR fences [{lower:.3f}, {upper:.3f}], flagged rows: {int(flagged.sum())}")
    outlier_rows.extend(
        {"date": date, "column": column, "value": value, "lower_fence": lower, "upper_fence": upper}
        for date, value in df.loc[flagged, column].items()
    )
display(pd.DataFrame(outlier_rows, columns=["date", "column", "value", "lower_fence", "upper_fence"]))

range_issues = {
    "meantemp outside -50..60 C (screening range)": ~df["meantemp"].between(-50, 60),
    "humidity outside 0..100%": ~df["humidity"].between(0, 100),
    "negative wind_speed": df["wind_speed"] < 0,
    "meanpressure outside 800..1100 hPa (screening range)": ~df["meanpressure"].between(800, 1100),
}
for label, mask in range_issues.items():
    print(f"{label}: {int(mask.sum())} row(s)")
    if mask.any():
        display(df.loc[mask])

## Make the cleaned working copy

The Jan 1 pressure value is replaced with `NaN` in memory only. It is not possible to recover the true pressure from this file alone, so no replacement value is fabricated. IQR-flagged wind speeds are retained. Missing values and missing dates remain visible and are not silently imputed.

In [ ]:
clean_df = df.copy()
pressure_anomalies = ~clean_df["meanpressure"].between(800, 1100)
print("Pressure values marked missing:")
display(clean_df.loc[pressure_anomalies, ["meanpressure"]])
clean_df.loc[pressure_anomalies, "meanpressure"] = np.nan

print("Missing values in cleaned data:")
display(clean_df.isna().sum().rename("missing values"))
print("Cleaned summary:")
display(clean_df.describe())

if missing_dates.size:
    raise ValueError("The daily series has calendar gaps; resolve them explicitly before ARIMA forecasting.")
if clean_df["meantemp"].isna().any():
    raise ValueError("Mean temperature contains missing values; resolve them explicitly before forecasting.")
temperatures = clean_df["meantemp"].asfreq("D")

In [ ]:
ax = temperatures.plot(figsize=(12, 4), title="Delhi daily mean temperature")
ax.set_xlabel("Date")
ax.set_ylabel("Mean temperature (C)")
plt.tight_layout()
plt.show()

## Chronological holdout evaluation

The final 30 observations are held out so evaluation respects time order.

In [ ]:
evaluate_arima(temperatures, test_size=30)

## Seven-day forecast

In [ ]:
forecast_temperatures(temperatures, steps=7)